In [2]:
import pandas as pd
import numpy as np
import urllib.request
import zipfile
import io

# Download Online Retail II
url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"

print("Downloading Online Retail II dataset...")
with urllib.request.urlopen(url) as response:
    with zipfile.ZipFile(io.BytesIO(response.read())) as z:
        with z.open('online_retail_II.xlsx') as f:
            df = pd.read_excel(f, sheet_name='Year 2010-2011')

print(f"Raw data: {len(df):,} rows")
df.head()

Raw data: 541,910 rows


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Clean and Prepare Data

In [5]:
# Remove missing customers
df = df[df['Customer ID'].notna()]

# Keep only positive quantities (purchases, not returns)
df = df[df['Quantity'] > 0]

# Keep only positive prices
df = df[df['Price'] > 0]

# Remove cancelled invoices (start with C)
df = df[~df['Invoice'].astype(str).str.startswith('C')]

print(f"Cleaned data: {len(df):,} rows")
print(f"Customers: {df['Customer ID'].nunique():,}")
print(f"Products: {df['StockCode'].nunique():,}")
print(f"Countries: {df['Country'].nunique()}")

Cleaned data: 397,885 rows
Customers: 4,338
Products: 3,665
Countries: 37


## Create Customer-Product Matrix

In [7]:
# Create binary matrix (1 if customer bought product, 0 otherwise)
customer_product = df.groupby(['Customer ID', 'StockCode']).size().unstack().fillna(0)
customer_binary = (customer_product > 0).astype(int)
print(f"Customer-Product Matrix: {customer_binary.shape[0]} customers × {customer_binary.shape[1]} products")
print(f"Sparsity: {(1 - customer_binary.sum().sum() / (customer_binary.shape[0] * customer_binary.shape[1])):.2%}")

Customer-Product Matrix: 4338 customers × 3665 products
Sparsity: 98.32%


## Build Customer Similarity Graph

In [9]:
from sklearn.metrics import jaccard_score
import networkx as nx

n_customers = customer_binary.shape[0]
similarity_matrix = np.zeros((n_customers, n_customers))

print("Computing customer similarities...")
for i in range(n_customers):
    if i % 500 == 0:
        print(f"  Processing customer {i}/{n_customers}")
    for j in range(i+1, n_customers):
        sim = jaccard_score(customer_binary.iloc[i], customer_binary.iloc[j])
        if sim > 0.3:  # Similarity threshold
            similarity_matrix[i, j] = sim
            similarity_matrix[j, i] = sim

# Build graph
G = nx.Graph()
G.add_nodes_from(range(n_customers))

for i in range(n_customers):
    for j in range(i+1, n_customers):
        if similarity_matrix[i, j] > 0:
            G.add_edge(i, j, weight=similarity_matrix[i, j])

print(f"\nGraph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Average degree: {2 * G.number_of_edges() / G.number_of_nodes():.2f}")